In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
torch.backends.cudnn.benchmark = True

In [3]:
train_dir = "/kaggle/input/datasets/priyaadharshinivs062/leukemia-dataset/train/train"
test_dir  = "/kaggle/input/datasets/priyaadharshinivs062/leukemia-dataset/test/test"

In [4]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [5]:
from torchvision.datasets import ImageFolder

class FilteredImageFolder(ImageFolder):
    def __init__(self, root, transform=None, exclude_classes=None):
        self.exclude_classes = exclude_classes or []
        super().__init__(root, transform=transform)

    def find_classes(self, directory):
        classes = sorted(entry.name for entry in os.scandir(directory) if entry.is_dir())
        
        # ❌ Remove unwanted classes
        classes = [cls for cls in classes if cls not in self.exclude_classes]
        
        class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
        return classes, class_to_idx

In [6]:
exclude = ['h train']

train_dataset = FilteredImageFolder(
    train_dir,
    transform=train_transforms,
    exclude_classes=exclude
)

test_dataset = FilteredImageFolder(
    test_dir,
    transform=test_transforms,
    exclude_classes=exclude
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print("Classes USED:", train_dataset.classes)

Classes USED: ['all train', 'aml train', 'cll train', 'cml']


In [7]:
model = models.efficientnet_b0(pretrained=True)

# Freeze all feature layers
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier
num_classes = len(train_dataset.classes)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

model = model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 189MB/s]


In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [9]:
def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    print(classification_report(all_labels, all_preds))

In [10]:
from tqdm import tqdm
import torch

def train_model(model, train_loader, test_loader, epochs=20):
    global optimizer

    scaler = torch.cuda.amp.GradScaler()  # 🔥 for mixed precision

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        print(f"\n📌 Epoch [{epoch+1}/{epochs}]")

        # 🔓 Unfreeze after epoch 8
        if epoch == 8:
            print("🔥 Unfreezing last layers...")
            for param in model.features[-2:].parameters():
                param.requires_grad = True

            optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

        loop = tqdm(train_loader, desc="Training", leave=False)

        for images, labels in loop:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()

            # ⚡ Mixed Precision Forward
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            # ⚡ Backward with scaler
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()

            # 📊 Accuracy
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            # 🔄 Update tqdm
            loop.set_postfix(
                loss=loss.item(),
                acc=100 * correct / total
            )

        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100 * correct / total

        print(f"✅ Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}%")

        # 📊 Validation
        evaluate(model, test_loader)

In [11]:
train_model(model, train_loader, test_loader, epochs=25)

/tmp/ipykernel_23/1382637046.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()  # 🔥 for mixed precision



📌 Epoch [1/25]


Training:   0%|          | 0/375 [00:00<?, ?it/s]/tmp/ipykernel_23/1382637046.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
                                                                               

✅ Train Loss: 0.5631 | Train Acc: 79.99%


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


...

              precision    recall  f1-score   support

           0       0.54      0.42      0.47      1000
           1       0.69      0.89      0.78      1000
           2       0.52      0.91      0.66      1000
           3       0.83      0.98      0.90      1000
           4       0.00      0.00      0.00      1000

    accuracy                           0.64      5000
   macro avg       0.52      0.64      0.56      5000
weighted avg       0.52      0.64      0.56      5000


📌 Epoch [25/25]


Training:   0%|          | 0/375 [00:00<?, ?it/s]/tmp/ipykernel_23/1382637046.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
                                                                                 

✅ Train Loss: 0.0923 | Train Acc: 96.69%


              precision    recall  f1-score   support

           0       0.55      0.45      0.50      1000
           1       0.71      0.88      0.78      1000
           2       0.51      0.92      0.66      1000
           3       0.86      0.98      0.92      1000
           4       0.00      0.00      0.00      1000

    accuracy                           0.65      5000
   macro avg       0.53      0.65      0.57      5000
weighted avg       0.53      0.65      0.57      5000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [12]:
torch.save(model.state_dict(), "/kaggle/working/leukemia_model.pth")